# openSMILE Feature Sets (IEMOCAP)

This notebook extracts standard openSMILE feature sets (eGeMAPS/GeMAPS/ComParE).
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path

import librosa
import pandas as pd


In [2]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "smilesets"
OUT_FILE = "smilesets_features.csv"

# openSMILE params
FEATURE_SET = "eGeMAPSv02"  # options: GeMAPSv01b, eGeMAPSv02, ComParE_2016

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


In [3]:
_SMILE_CACHE: dict[str, object] = {}


def _require_opensmile():
    # Import opensmile with a clear error if missing
    try:
        import opensmile
    except ImportError as exc:
        raise ImportError(
            "openSMILE features require the opensmile package. "
            "Install with: uv add opensmile"
        ) from exc
    return opensmile


def _get_smile(feature_set: str):
    # Load or reuse a Smile extractor
    cached = _SMILE_CACHE.get(feature_set)
    if cached is not None:
        return cached

    opensmile = _require_opensmile()
    feature_map = {
        "GeMAPSv01b": opensmile.FeatureSet.GeMAPSv01b,
        "eGeMAPSv02": opensmile.FeatureSet.eGeMAPSv02,
        "ComParE_2016": opensmile.FeatureSet.ComParE_2016,
    }
    if feature_set not in feature_map:
        valid = ", ".join(sorted(feature_map))
        raise ValueError(f"Unknown feature_set '{feature_set}'. Valid: {valid}")

    smile = opensmile.Smile(
        feature_set=feature_map[feature_set],
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    _SMILE_CACHE[feature_set] = smile
    return smile


def extract_smile(audio_path: Path, *, feature_set: str) -> dict[str, float]:
    smile = _get_smile(feature_set)
    features = smile.process_file(str(audio_path))

    flat: dict[str, float] = {}
    for name, value in features.iloc[0].items():
        flat[f"smile_{name}"] = float(value)
    return flat


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


In [5]:
rows: list[dict[str, float | str | int]] = []
missing: list[str] = []

for _, row in df.iterrows():
    rel_path = row["path"]
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        missing.append(str(audio_path))
        continue

    duration_s = float(librosa.get_duration(filename=audio_path))
    features = extract_smile(audio_path, feature_set=FEATURE_SET)

    record: dict[str, float | str | int] = {
        "path": str(rel_path),
        "session": int(row["session"]),
        "method": row["method"],
        "gender": row["gender"],
        "emotion": row["emotion"],
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    rows.append(record)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape
